# Full benchmark: all 18 datasets, all methods, all statistical tests

Runs everything end to end and stores it under `results/<EXPERIMENT_NAME>/`:

1. **Main experiment**: K-Means, FCM, EAC, ACE, Fuzzy-ACE × 18 datasets (12 UCI + 6 SIPU) × {fixed, variable} k × `N_RUNS` seeded runs, scored with ARI and NMI.
2. **Statistical tests**: Friedman + Iman-Davenport, then Nemenyi post-hoc and CD diagrams, on all datasets and on three subsets (ACE paper, fuzzy hypothesis, SIPU).
3. **α₂ sensitivity**: grid search of the certainty threshold for ACE and Fuzzy-ACE.

Every finished run is written to disk immediately, so re-running the notebook after an interruption resumes where it left off.
Birch1-3 (n = 100,000) are replaced by one fixed stratified subsample of `MAX_INSTANCES` points (EAC needs an n × n matrix).
The full run takes hours; set `QUICK = True` first to check that everything works in a few minutes.

In [1]:
%load_ext autoreload
%autoreload 2

import sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.datasets import (ACE_PAPER_DATASETS, ALL_DATASETS, FUZZY_HYPOTHESIS_DATASETS, SIPU_DATASETS,
                          dataset_overview)
from src.experiments import METHODS, ExperimentConfig, run_alpha2_grid_search, run_experiment
from src.stats_tests import run_statistical_tests

pd.set_option("display.precision", 4)
pd.set_option("display.width", 200)
RESULTS_DIR = ROOT / "results"
print("Project root:", ROOT)

Project root: C:\Users\lukas\PycharmProjects\FuzzyAdaptiveClusteringEnsemble


## Parameters

In [2]:
QUICK = False                  # True: 2 runs on 4 small datasets, to test the pipeline quickly

EXPERIMENT_NAME = "full"
DATASETS = list(ALL_DATASETS)
N_RUNS = 10
N_JOBS = -1                    # parallel workers (-1 = all cores); lower it if memory runs out
MAX_INSTANCES = 5000           # Birch subsample size (None = all 100,000 points; not feasible for EAC)
VERBOSE = True                 # print one line per finished run

STATS_ALPHA = 0.1
STATS_SUBSETS = {"": DATASETS, "ace_paper": ACE_PAPER_DATASETS,
                 "fuzzy_hypothesis": FUZZY_HYPOTHESIS_DATASETS, "sipu": SIPU_DATASETS}

ALPHA2_VALUES = [0.5, 0.6, 0.7, 0.8, 0.9]
ALPHA2_DATASETS = list(ALL_DATASETS)
ALPHA2_SETTINGS = ["fixed", "variable"]

if QUICK:
    EXPERIMENT_NAME, N_RUNS = "full_quick", 2
    DATASETS = ALPHA2_DATASETS = ["iris", "wine", "unbalance", "dim032"]
    STATS_SUBSETS = {"": DATASETS}

cfg = ExperimentConfig(name=EXPERIMENT_NAME, datasets=DATASETS, n_runs=N_RUNS, n_jobs=N_JOBS,
                       max_instances=MAX_INSTANCES, results_dir=str(RESULTS_DIR))
exp_dir = RESULTS_DIR / EXPERIMENT_NAME
grid_dir = RESULTS_DIR / f"{EXPERIMENT_NAME}_alpha2"
cfg

ExperimentConfig(name='full', datasets=['iris', 'wine', 'new_thyroid', 'glass', 'breast_cancer', 'soybean_small', 'ionosphere', 'ecoli', 'yeast', 'vehicle', 'image_segmentation', 'dermatology', 'birch1', 'birch2', 'birch3', 'unbalance', 'dim032', 'dim064'], settings=['fixed', 'variable'], methods=['KMeans', 'FCM', 'EAC', 'ACE', 'Fuzzy-ACE'], n_runs=10, base_seed=0, m_partitions=10, sampling_ratio=0.7, alpha_1=0.8, alpha_2=0.7, delta_alpha=0.1, fcm_m_fuzz=2.0, fuzzy_ace_m_range=(1.5, 2.5), kmeans_n_init=1, eac_linkage='average', max_instances=5000, subsample_seed=0, data_dir='C:\\Users\\lukas\\PycharmProjects\\FuzzyAdaptiveClusteringEnsemble\\data\\raw', results_dir='C:\\Users\\lukas\\PycharmProjects\\FuzzyAdaptiveClusteringEnsemble\\results', n_jobs=-1)

## 1. Datasets

In [3]:
overview = dataset_overview().set_index("key").loc[DATASETS]
overview["n_used"] = [min(n, MAX_INSTANCES) if MAX_INSTANCES else n for n in overview["n"]]
overview["group"] = ["ACE paper" if k in ACE_PAPER_DATASETS else
                     "fuzzy hypothesis" if k in FUZZY_HYPOTHESIS_DATASETS else "SIPU" for k in overview.index]
missing = overview.index[~overview["available"]].tolist()
if missing:
    print("Missing raw files (these will be skipped):", missing, "-> run: bash data/download_uci.sh")
overview

,name,available,n,d,k,missing_values,n_used,group
key,,,,,,,,
iris,Iris,True,150,4,3,0,150,ACE paper
wine,Wine,True,178,13,3,0,178,ACE paper
new_thyroid,New-Thyroid,True,215,5,3,0,215,ACE paper
glass,Glass,True,214,9,6,0,214,ACE paper
breast_cancer,Breast Cancer (Original),True,699,9,2,16,699,ACE paper
soybean_small,Soybean (Small),True,47,35,4,0,47,ACE paper
ionosphere,Ionosphere,True,351,34,2,0,351,ACE paper
ecoli,Ecoli,True,336,7,8,0,336,fuzzy hypothesis
yeast,Yeast,True,1484,8,10,0,1484,fuzzy hypothesis


## 2. Main experiment

In [4]:
t0 = time.perf_counter()
raw = run_experiment(cfg, resume=True, verbose=VERBOSE)
print(f"{len(raw)} runs stored, {int((raw['status'] != 'ok').sum())} failed, "
      f"this session {(time.perf_counter() - t0) / 60:.1f} min")

[full] 1800 runs to do (0 already stored) -> C:\Users\lukas\PycharmProjects\FuzzyAdaptiveClusteringEnsemble\results\full
  [1/1800] fixed    iris               KMeans    run 5: ARI=0.6201 NMI=0.6595 (0.5s) ok
  [2/1800] fixed    iris               KMeans    run 8: ARI=0.5923 NMI=0.6427 (0.0s) ok
  [3/1800] fixed    iris               KMeans    run 9: ARI=0.4290 NMI=0.5874 (0.0s) ok
  [4/1800] fixed    iris               KMeans    run 4: ARI=0.6410 NMI=0.6728 (0.4s) ok
  [5/1800] fixed    iris               FCM       run 0: ARI=0.6303 NMI=0.6659 (0.1s) ok
  [6/1800] fixed    iris               FCM       run 1: ARI=0.6303 NMI=0.6659 (0.0s) ok
  [7/1800] fixed    iris               FCM       run 2: ARI=0.6303 NMI=0.6659 (0.0s) ok
  [8/1800] fixed    iris               FCM       run 3: ARI=0.6303 NMI=0.6659 (0.0s) ok
  [9/1800] fixed    iris               FCM       run 4: ARI=0.6303 NMI=0.6659 (0.0s) ok
  [10/1800] fixed    iris               FCM       run 5: ARI=0.6303 NMI=0.6659 (0.0s) o

KeyboardInterrupt: 

In [ ]:
failed = raw[raw["status"] != "ok"]
if len(failed):
    display(failed[["setting", "dataset", "method", "run", "error"]])
else:
    print("No failed runs.")

### Mean ± std per dataset (last row: average over datasets)

In [ ]:
for setting in cfg.settings:
    for metric in ("ari", "nmi"):
        print(f"\n{metric.upper()} - {setting} k")
        display(pd.read_csv(exp_dir / "tables" / f"{metric}_{setting}.csv", index_col=0))

### Mean runtime per run [s]

In [ ]:
ok = raw[raw["status"] == "ok"]
runtime = ok.groupby(["dataset", "method"])["runtime_s"].mean().unstack()
runtime.loc[[d for d in DATASETS if d in runtime.index], [m for m in METHODS if m in runtime.columns]]

### Spread over the runs

In [ ]:
present = [d for d in DATASETS if d in raw["dataset"].unique()]
n_cols = 6
n_rows = int(np.ceil(len(present) / n_cols))
for setting in cfg.settings:
    for metric in ("ari", "nmi"):
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 2.8 * n_rows), squeeze=False)
        for ax, d in zip(axes.flat, present):
            sub = ok[(ok.setting == setting) & (ok.dataset == d)]
            ax.boxplot([sub[sub.method == m][metric] for m in METHODS], tick_labels=list(METHODS))
            ax.set_title(d, fontsize=10)
            ax.tick_params(axis="x", labelrotation=45, labelsize=8)
            ax.grid(alpha=0.3)
        for ax in axes.flat[len(present):]:
            ax.axis("off")
        fig.suptitle(f"{metric.upper()} - {setting} k")
        fig.tight_layout()
        plt.show()

## 3. Statistical tests (Iman-Davenport + Nemenyi, CD diagrams)

Results of the subsets go to `statistics_<subset>/`; all datasets go to `statistics/`.

In [ ]:
for tag, subset in STATS_SUBSETS.items():
    subset = [d for d in subset if d in present]
    if len(subset) < 2:
        continue
    print(f"\n=== {tag or 'all datasets'} ({len(subset)} datasets)")
    for r in run_statistical_tests(str(exp_dir), alpha=STATS_ALPHA, datasets=subset, tag=tag):
        verdict = "REJECT H0" if r.reject_h0 else "no significant difference"
        pairs = "; ".join(f"{a} > {b}" for a, b, _ in r.significant_pairs) or "none"
        print(f"{r.metric.upper():3s} {r.setting:8s} Iman-Davenport p={r.iman_davenport_p:.4f} -> {verdict}; "
              f"CD={r.critical_difference:.3f}; significant: {pairs}")
        print("    average ranks:", ", ".join(f"{m} {v:.2f}" for m, v in r.average_ranks.items()))

In [ ]:
display(pd.read_csv(exp_dir / "statistics" / "tests_summary.csv"))
for png in sorted((exp_dir / "statistics" / "cd_diagrams").glob("*.png")):
    display(Image(filename=str(png)))

## 4. α₂ sensitivity (ACE and Fuzzy-ACE)

Stages 1-2 are fitted once per run; only Stage 3 is repeated for each α₂ value.

In [ ]:
grid_cfg = ExperimentConfig(name=grid_dir.name, datasets=ALPHA2_DATASETS, settings=ALPHA2_SETTINGS,
                            methods=["ACE", "Fuzzy-ACE"], n_runs=N_RUNS, n_jobs=N_JOBS,
                            max_instances=MAX_INSTANCES, results_dir=str(RESULTS_DIR))
grid_raw = run_alpha2_grid_search(grid_cfg, alpha_2_values=ALPHA2_VALUES, verbose=VERBOSE)
display(pd.read_csv(grid_dir / "average_ranks.csv"))
pd.read_csv(grid_dir / "stability.csv").query("metric == 'ari'")

In [ ]:
for png in sorted((grid_dir / "figures").glob("*.png")):
    display(Image(filename=str(png)))

## 5. Stored files

In [ ]:
for d in (exp_dir, grid_dir):
    print(d.relative_to(ROOT))
    for p in sorted(d.rglob("*")):
        if p.is_file():
            print("   ", p.relative_to(d))